# Create Datasets

- Use to split data signals into separate files.

In [1]:
import os
import sys

import pandas as pd

notebook_dir = os.getcwd()

sys.path.append(os.path.join(notebook_dir, '../python/lamina/'))

from data_processing import DataProcessing

## Load Data

In [ ]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
data_path = os.path.join(base_data_path, 'WESAD/all_subjects/chest.csv')
df = DataProcessing.load_from_file(data_path, file_type='csv')
df.drop(['Unnamed: 0'], axis=1, inplace=True)
df

,ACC_x,ACC_y,ACC_z,BVP,EDA,Temp,subject
0,62.0,-21.0,107.0,-59.37,1.138257,35.41,S2
1,66.0,13.0,53.0,-53.42,1.125444,35.41,S2
2,41.0,9.0,15.0,-44.40,1.011405,35.41,S2
3,52.0,16.0,24.0,-33.17,1.033188,35.41,S2
4,54.0,15.0,34.0,-20.79,0.935807,35.41,S2
...,...,...,...,...,...,...,...
5559547,NaN,NaN,NaN,-2.09,NaN,NaN,S17
5559548,NaN,NaN,NaN,-3.28,NaN,NaN,S17
5559549,NaN,NaN,NaN,-4.43,NaN,NaN,S17
5559550,NaN,NaN,NaN,-5.44,NaN,NaN,S17


In [3]:
copy_df = df.copy()
copy_df

,ACC_x,ACC_y,ACC_z,BVP,EDA,Temp,subject
0,62.0,-21.0,107.0,-59.37,1.138257,35.41,S2
1,66.0,13.0,53.0,-53.42,1.125444,35.41,S2
2,41.0,9.0,15.0,-44.40,1.011405,35.41,S2
3,52.0,16.0,24.0,-33.17,1.033188,35.41,S2
4,54.0,15.0,34.0,-20.79,0.935807,35.41,S2
...,...,...,...,...,...,...,...
5559547,NaN,NaN,NaN,-2.09,NaN,NaN,S17
5559548,NaN,NaN,NaN,-3.28,NaN,NaN,S17
5559549,NaN,NaN,NaN,-4.43,NaN,NaN,S17
5559550,NaN,NaN,NaN,-5.44,NaN,NaN,S17


## Keep Meaningful Labels

- 0 = not defined / transient
- 1 = baseline
- 2 = stress
- 3 = amusement
- 4 = meditation
- 5/6/7 = should be ignored in this dataset

In [4]:
filt_labels = copy_df['label'].isin([0, 5, 6, 7])
copy_df = copy_df[~filt_labels]
copy_df

KeyError: 'label'

## Signal of Interest

In [5]:
signal_df = copy_df.loc[:, ['ECG', 'subject', 'label']]
signal_df

,ECG,subject,label
214583,0.030945,S2,1
214584,0.033646,S2,1
214585,0.033005,S2,1
214586,0.031815,S2,1
214587,0.030350,S2,1
...,...,...,...
60620859,0.195236,S17,4
60620860,0.178574,S17,4
60620861,0.158020,S17,4
60620862,0.129959,S17,4


## Clean Data

In [6]:
cleaned_signal_df = DataProcessing.get_cleaned_data(signal_df, 'ECG', 700)
cleaned_signal_df

,ECG,subject,label,Cleaned ECG
214583,0.030945,S2,1,0.006982
214584,0.033646,S2,1,0.007554
214585,0.033005,S2,1,0.008133
214586,0.031815,S2,1,0.008776
214587,0.030350,S2,1,0.009535
...,...,...,...,...
60620859,0.195236,S17,4,0.270232
60620860,0.178574,S17,4,0.256066
60620861,0.158020,S17,4,0.241534
60620862,0.129959,S17,4,0.226662


## Set each subject with new row 0

In [7]:
new_signal_df = DataProcessing.reset_rows(cleaned_signal_df)
new_signal_df

,ECG,subject,label,Cleaned ECG
0,0.030945,S2,1,0.006982
1,0.033646,S2,1,0.007554
2,0.033005,S2,1,0.008133
3,0.031815,S2,1,0.008776
4,0.030350,S2,1,0.009535
...,...,...,...,...
2104895,0.195236,S17,4,0.270232
2104896,0.178574,S17,4,0.256066
2104897,0.158020,S17,4,0.241534
2104898,0.129959,S17,4,0.226662


### Ex to get specific subject

In [8]:
s_df = DataProcessing.get_specific_subject_df(new_signal_df, 'S3')
s_df

,ECG,subject,label,Cleaned ECG
0,-0.191849,S3,1,-0.176445
1,-0.186630,S3,1,-0.176682
2,-0.179260,S3,1,-0.176798
3,-0.171432,S3,1,-0.176842
4,-0.173264,S3,1,-0.176896
...,...,...,...,...
2054496,0.048477,S3,4,0.067996
2054497,0.035019,S3,4,0.065061
2054498,0.026917,S3,4,0.062005
2054499,0.019455,S3,4,0.058963


In [26]:
def check_order(s_df):
    # s is a Series here
    s = s_df.loc[s_df['label'].isin([3, 4]), 'label']

    run_starts = s[s.ne(s.shift())]
    run_order = run_starts.to_numpy()

    print(type(run_order))
    return run_order

In [27]:
labels = new_signal_df['subject'].unique()
for label in labels:
    # print(label)

    s_df = DataProcessing.get_specific_subject_df(new_signal_df, label)
    order = check_order(s_df)
    print(order)
    import numpy as np
    normal = np.array_equal(order, [3, 4])
    abnormal = np.array_equal(order, [4, 3, 4])

    print(f"Normal: {normal}")
    print(f"Abnormal: {abnormal}")

<class 'numpy.ndarray'>
[4 3 4]
Normal: False
Abnormal: True
<class 'numpy.ndarray'>
[4 3 4]
Normal: False
Abnormal: True
<class 'numpy.ndarray'>
[3 4]
Normal: True
Abnormal: False
<class 'numpy.ndarray'>
[3 4]
Normal: True
Abnormal: False
<class 'numpy.ndarray'>
[4 3 4]
Normal: False
Abnormal: True
<class 'numpy.ndarray'>
[3 4]
Normal: True
Abnormal: False
<class 'numpy.ndarray'>
[3 4]
Normal: True
Abnormal: False
<class 'numpy.ndarray'>
[4 3 4]
Normal: False
Abnormal: True
<class 'numpy.ndarray'>
[3 4]
Normal: True
Abnormal: False
<class 'numpy.ndarray'>
[4 3 4]
Normal: False
Abnormal: True
<class 'numpy.ndarray'>
[3 4]
Normal: True
Abnormal: False
<class 'numpy.ndarray'>
[4 3 4]
Normal: False
Abnormal: True
<class 'numpy.ndarray'>
[3 4]
Normal: True
Abnormal: False
<class 'numpy.ndarray'>
[4 3 4]
Normal: False
Abnormal: True
<class 'numpy.ndarray'>
[3 4]
Normal: True
Abnormal: False


## Save Data

In [11]:
save_data_path = os.path.join(notebook_dir, "../../data/WESAD/all_subjects/")
save_data = os.path.join(save_data_path, 'ecg_chest-subset_labels-cleaned.csv')
new_signal_df.to_csv(save_data, index=True)